# 05 — Cross-Encoder Reranking with BGE-M3

## Position in pipeline
```
PDF -> Chunks -> Embeddings -> [04 Retrieval] -> **[05 Reranking]** -> [06 LLM] -> Answer
```

## Purpose

Hybrid retrieval (BM25 + dense) returns a broad set of ~20 candidate documents.
Many are topically relevant but do not directly answer the query. A **cross-encoder
reranker** rescores every (query, document) pair through a single Transformer pass,
producing a fine-grained relevance score. This is more accurate than dual-encoder
cosine similarity but too slow to run over the full corpus, hence the two-stage
retrieve-then-rerank design.

We use **BAAI/bge-reranker-v2-m3**, a cross-encoder model trained on multilingual
retrieval benchmarks. It tokenizes the concatenated (query, passage) pair and outputs
a logit score where higher = more relevant.

## Learning objectives

1. Understand why reranking improves precision over first-stage retrieval
2. Load the BGE cross-encoder model and tokenizer
3. Score individual (query, doc) pairs and interpret the raw logit values
4. Rerank a candidate list and compare before vs. after ordering
5. Visualize the score distribution to see how well the model separates relevant from irrelevant passages

## Imports

We need `torch` for inference, `transformers` for the model and tokenizer, and
`langchain_core.documents.Document` to match the data structure used across the
pipeline. `matplotlib` is used for the score distribution chart.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from langchain_core.documents import Document
from src import constant

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Reranker model  : {constant.bge_reranker_model_path}")

## Load the reranker model

The `BGEM3ReRanker` class in `src/reranker/bge_m3_reranker.py` wraps model loading
and scoring. Here we load the raw model directly so you can see what happens under
the hood: the model is a `AutoModelForSequenceClassification` with a single output
logit per (query, passage) pair.

In [ ]:
model_path = constant.bge_reranker_model_path
max_length = 4096

print(f"Loading tokenizer from: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path)

print(f"Loading model from: {model_path}")
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()
model.half()   # fp16 for faster inference

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Score example (query, doc) pairs

The cross-encoder takes a concatenated (query, passage) input and produces a raw
logit score. There is no fixed threshold — scores are only meaningful relative to
each other within the same query. A higher logit means the model considers the
passage more relevant to the query.

In [ ]:
query = "What is the role of T helper cells in adaptive immunity?"

passages = [
    "T helper cells coordinate adaptive immune responses by secreting cytokines "
    "that activate B cells, cytotoxic T cells, and macrophages. They recognize "
    "antigens presented via MHC class II molecules.",

    "B cells produce antibodies in response to antigens. Upon activation, they "
    "differentiate into plasma cells that secrete large quantities of immunoglobulin.",

    "Natural killer cells are part of the innate immune system and can kill "
    "virus-infected cells without prior sensitization.",

    "CD4+ T helper cells are essential for orchestrating adaptive immunity. "
    "Th1 cells promote cell-mediated responses while Th2 cells drive humoral immunity.",

    "The complement system consists of serum proteins that enhance phagocytosis "
    "and can directly lyse pathogen membranes through the membrane attack complex.",
]

# Build (query, passage) pairs for the tokenizer
pairs = [(query, p) for p in passages]

inputs = tokenizer(
    pairs,
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=max_length,
).to(device)

with torch.no_grad():
    logits = model(**inputs).logits

scores = logits.detach().cpu().float().numpy().flatten()

print(f"Query: {query}\n")
print(f"{'Rank':>4}  {'Score':>8}  Passage (truncated)")
print("-" * 70)
for i, (score, passage) in enumerate(zip(scores, passages)):
    print(f"{i+1:>4}  {score:>8.3f}  {passage[:60]}...")

## Rank candidate documents — before vs. after reranking

In the real pipeline, the retriever returns documents in BM25/dense score order.
After cross-encoder reranking, the order may change significantly. Here we simulate
the full flow: create `Document` objects as the retriever would, score them with the
reranker, and compare the ordering before and after.

In [ ]:
# Simulate retriever output order (intentionally suboptimal)
candidate_docs = [
    Document(page_content=passages[2], metadata={"doc_id": "C", "title": "NK cells"}),
    Document(page_content=passages[1], metadata={"doc_id": "B", "title": "B cells"}),
    Document(page_content=passages[4], metadata={"doc_id": "E", "title": "Complement"}),
    Document(page_content=passages[0], metadata={"doc_id": "A", "title": "T helper overview"}),
    Document(page_content=passages[3], metadata={"doc_id": "D", "title": "Th1 vs Th2"}),
]

print("=== BEFORE reranking (retriever order) ===")
for i, doc in enumerate(candidate_docs, 1):
    print(f"  {i}. [{doc.metadata['doc_id']}] {doc.metadata['title']}")

# Score with cross-encoder
pairs = [(query, doc.page_content) for doc in candidate_docs]
inputs = tokenizer(
    pairs, padding=True, truncation=True,
    return_tensors="pt", max_length=max_length,
).to(device)

with torch.no_grad():
    rerank_scores = model(**inputs).logits.detach().cpu().float().numpy().flatten()

# Sort by score descending
top_k = 3
ranked = sorted(
    zip(rerank_scores, candidate_docs),
    key=lambda x: x[0],
    reverse=True,
)[:top_k]

print(f"\n=== AFTER reranking (top {top_k}) ===")
for i, (score, doc) in enumerate(ranked, 1):
    print(f"  {i}. [{doc.metadata['doc_id']}] {doc.metadata['title']}  (score: {score:.3f})")

print(f"\nThe reranker pushed the most relevant T-helper passages to the top,")
print(f"while the complement and NK cell passages (topically adjacent but less relevant) dropped.")

## Visualization — score distribution bar chart

A horizontal bar chart shows how the cross-encoder scores distribute across
candidates. A good reranker produces a clear gap between relevant and irrelevant
passages, making the top-k cutoff effective.

In [ ]:
labels = [doc.metadata["title"] for doc in candidate_docs]

# Sort for display
sorted_indices = np.argsort(rerank_scores)
sorted_labels = [labels[i] for i in sorted_indices]
sorted_scores = rerank_scores[sorted_indices]

# Color: green for top_k, gray for the rest
colors = []
top_k_ids = set(i for i, _ in sorted(
    enumerate(rerank_scores), key=lambda x: x[1], reverse=True
)[:top_k])
for idx in sorted_indices:
    colors.append("#2ecc71" if idx in top_k_ids else "#95a5a6")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(sorted_labels, sorted_scores, color=colors, edgecolor="white")
ax.set_xlabel("Cross-Encoder Logit Score")
ax.set_title(f"Reranker Score Distribution\nQuery: \"{query[:50]}...\"")

# Annotate scores on bars
for bar, score in zip(bars, sorted_scores):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.2f}",
        va="center", fontsize=9,
    )

ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, fc="#2ecc71", label=f"Top-{top_k} (kept)"),
        plt.Rectangle((0, 0), 1, 1, fc="#95a5a6", label="Filtered out"),
    ],
    loc="lower right",
)

plt.tight_layout()
plt.show()

print(f"The green bars are the top-{top_k} passages kept after reranking.")
print(f"The gray bars are filtered out before being sent to the LLM.")

## Summary

### What this notebook produced
- Loaded the BGE cross-encoder reranker and scored (query, passage) pairs
- Demonstrated before vs. after reranking order
- Visualized the score distribution across candidates

### Key takeaways
- Cross-encoder reranking is more accurate than dual-encoder retrieval but slower
- The two-stage retrieve-then-rerank pattern keeps latency manageable
- The reranker produces raw logit scores (no threshold needed, only relative ordering matters)
- In production, the `BGEM3ReRanker` class in `src/reranker/bge_m3_reranker.py` wraps this logic

### Next notebook
**`06_llm.ipynb`** — LLM inference: system prompts, streaming, multi-turn, and HyDE query expansion